# Fixed positive kernel operator — Notebook 17 reusable definitions

Purpose: test a batched positive feature expansion of raw ViT content interactions, with exact replacement in a predeclared 3×3 patch neighborhood. Inputs are actual full-width Q/K/V, masks and original patch indices; outputs have the same token shape. This is an approximate attention hypothesis, not an accuracy or FMM guarantee.

All definition cells are explicitly `ahr-export`. Run All defines APIs only and takes seconds. It does not train or access data. Frozen feature matrices are model buffers in the containing run's full checkpoints; exact resume uses that run's manifest and notebook launch cell. Rank and feature seed must be registered before development evaluation. The short validation companion is `tests/17_Operator_Validation.ipynb`.

Random-feature identity: for Gaussian ω, E[exp(ω·x−||x||²/2) exp(ω·y−||y||²/2)] = exp(x·y), with x=q/√τ and y=k/√τ. Orthogonal directions preserve Gaussian row marginals while correlating features within a block. Finite normalized outputs are approximate and not unbiased simply because unnormalized entries have the expectation above. See [Performer](https://arxiv.org/abs/2009.14794).

Numerical policy: one shared key shift per sample/head and one query-row shift cancel from normalization. Exact local scores use the same scales. No independent key shifts or denominator clamping. Unsupported underflow/cancellation yields an explicit error. No dense far interaction matrix is built by the fast operator; dense matrices occur only in the separately named validation reference.

In [ ]:
from pathlib import Path
import json,os
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
_boot={'__file__':str(ROOT/'fmm_jepa/bootstrap.ipynb')}
for _cell in json.loads((ROOT/'fmm_jepa/bootstrap.ipynb').read_text())['cells']:
    if _cell['cell_type']=='code' and 'ahr-export' in _cell.get('metadata',{}).get('tags',[]):
        exec(compile(''.join(_cell['source']),'bootstrap#ahr-export','exec'),_boot)
_boot['install_notebook_package'](ROOT)


In [ ]:
"""Fixed positive random kernel features with exact local replacements.

This approximates content-only exponential dot-product attention. It is not
exact ViT attention, a novel FMM theorem, or a learned manifold certificate.
"""
import copy,hashlib,math
import torch
from torch import nn
from torch.nn import functional as F

BACKEND='fixed-positive-orthogonal-random-features-exact-local-v1'
ALLOWED_RANKS=(32,64,128)


def make_orthogonal_features(heads,head_dim,rank,seed,dtype=torch.float64):
    """Frozen Gaussian-marginal orthogonal blocks, independent CPU RNG only.

    Each row has a Haar direction and independent chi_D radius, giving the
    marginal N(0,I). Rows within a D-row block are dependent/orthogonal; extra
    blocks are independently drawn. No project/global RNG is consumed.
    """
    if min(int(heads),int(head_dim),int(rank))<1:raise ValueError('Positive feature dimensions required')
    generator=torch.Generator(device='cpu').manual_seed(int(seed));result=[]
    for head in range(int(heads)):
        blocks=[];remaining=int(rank)
        while remaining:
            gaussian=torch.randn(int(head_dim),int(head_dim),generator=generator,dtype=torch.float64)
            orthogonal,triangular=torch.linalg.qr(gaussian)
            signs=torch.where(triangular.diag()<0.,-torch.ones(head_dim,dtype=torch.float64),torch.ones(head_dim,dtype=torch.float64))
            directions=(orthogonal*signs[None]).T
            radii=torch.randn(head_dim,head_dim,generator=generator,dtype=torch.float64).norm(dim=-1)
            take=min(remaining,int(head_dim));blocks.append(directions[:take]*radii[:take,None]);remaining-=take
        result.append(torch.cat(blocks))
    return torch.stack(result).to(dtype=dtype)


def basis_sha256(feature_matrix):
    value=feature_matrix.detach().cpu().contiguous()
    header=(str(value.dtype)+':'+str(tuple(value.shape))).encode()
    return hashlib.sha256(header+value.numpy().tobytes()).hexdigest()


def local_neighbor_indices(indices,valid,grid,local_radius=1):
    """Predeclared grid neighborhood, preserving original indices and padding.

    Topology uses patch positions, not learned values or labels. Invalid token
    positions can be -1 and never overwrite a valid lookup. Repeated valid
    indices are refused. radius -1 explicitly disables local replacement.
    """
    if indices.ndim!=2 or valid.shape!=indices.shape or indices.dtype!=torch.long or valid.dtype!=torch.bool:
        raise ValueError('Expected long indices[B,N] and boolean valid[B,N]')
    height,width=map(int,grid);B,N=indices.shape;radius=int(local_radius)
    if height<1 or width<1 or radius not in(-1,0,1):raise ValueError('Valid grid and radius -1,0,1 required')
    if bool(((indices<0)|(indices>=height*width))[valid].any()):raise ValueError('Valid original patch index outside grid')
    if radius<0:return indices.new_zeros((B,N,0)),valid.new_zeros((B,N,0))
    positions=indices.clamp(0,height*width-1);lookup=indices.new_full((B,height*width),-1)
    batch=torch.arange(B,device=indices.device)[:,None].expand(B,N);slots=torch.arange(N,device=indices.device)[None].expand(B,N)
    counts=indices.new_zeros((B,height*width));counts.scatter_add_(1,positions,valid.long())
    if bool((counts>1).any()):raise ValueError('Duplicate valid original token index')
    lookup[batch[valid],positions[valid]]=slots[valid]
    row,col=positions//width,positions%width;neighbors=[];masks=[]
    for dr in range(-radius,radius+1):
        for dc in range(-radius,radius+1):
            rr,cc=row+dr,col+dc;inside=(rr>=0)&(rr<height)&(cc>=0)&(cc<width)&valid
            original=(rr.clamp(0,height-1)*width+cc.clamp(0,width-1));candidate=lookup.gather(1,original)
            mask=inside&(candidate>=0);neighbors.append(candidate.clamp_min(0));masks.append(mask)
    return torch.stack(neighbors,-1),torch.stack(masks,-1)


def _validate(q,k,v,valid,feature_matrix,tau):
    if q.ndim!=4 or k.shape!=q.shape or v.shape!=q.shape:raise ValueError('Matching q/k/v[B,H,N,D] required')
    B,H,N,D=q.shape
    if valid.shape!=(B,N) or valid.dtype!=torch.bool or valid.device!=q.device:raise ValueError('Boolean validity[B,N] on input device required')
    if k.device!=q.device or v.device!=q.device or k.dtype!=q.dtype or v.dtype!=q.dtype:raise ValueError('q/k/v dtype/devices differ')
    if q.dtype not in(torch.float32,torch.float64):raise ValueError('Validated arithmetic is FP32/FP64 only')
    if feature_matrix is None or feature_matrix.ndim!=3 or feature_matrix.shape[0]!=H or feature_matrix.shape[2]!=D:
        raise ValueError('Frozen feature matrix[H,R,D] required')
    W=feature_matrix.to(device=q.device,dtype=q.dtype)
    if not bool(torch.isfinite(W).all()):raise ValueError('Finite frozen feature matrix required')
    temperature=torch.full((H,),math.sqrt(D),device=q.device,dtype=q.dtype)if tau is None else torch.as_tensor(tau,device=q.device,dtype=q.dtype).reshape(H)
    if not bool(torch.isfinite(temperature).all())or bool((temperature<=0).any()):raise ValueError('Finite positive temperature required')
    mask=valid[:,None,:,None];safe=[torch.where(mask,t,torch.zeros_like(t))for t in(q,k,v)]
    if not all(bool(torch.isfinite(t).all())for t in safe):raise ValueError('Finite valid q/k/v required')
    return (*safe,W,temperature)


def _gather_tokens(value,neighbors):
    B,H,N,D=value.shape
    return value.gather(2,neighbors[:,None,:,None].expand(B,H,N,D))


def _log_features(q,k,W,tau,valid):
    scale=tau.sqrt()[None,:,None,None];x,y=q/scale,k/scale
    logq=torch.einsum('bhnd,hrd->bhnr',x,W)-.5*x.square().sum(-1,keepdim=True)
    logk=torch.einsum('bhnd,hrd->bhnr',y,W)-.5*y.square().sum(-1,keepdim=True)
    # ONE key shift per sample/head. A different shift for each key would alter
    # attention weights unless compensated, which this operator never does.
    masked=logk.masked_fill(~valid[:,None,:,None],-torch.inf)
    key_shift=masked.amax(dim=(-2,-1));key_shift=torch.where(valid.any(-1)[:,None],key_shift,torch.zeros_like(key_shift)).detach()
    return logq,logk,key_shift


def _feature_values(logq,logk,key_shift,valid,local_scores=None):
    query_shift=logq.amax(-1)
    if local_scores is not None and local_scores.shape[-1]:
        # Shared row shift also bounds exact local exponentials. It is not a
        # per-key shift and cancels from numerator and denominator alike.
        query_shift=torch.maximum(query_shift,local_scores.detach().amax(-1)-key_shift[:,:,None])
    query_shift=query_shift.detach();rank=logq.shape[-1];normalizer=math.sqrt(rank)
    phi_q=torch.exp(logq-query_shift[...,None])/normalizer
    phi_k=torch.exp(logk-key_shift[:,:,None,None])/normalizer
    mask=valid[:,None,:,None]
    return torch.where(mask,phi_q,torch.zeros_like(phi_q)),torch.where(mask,phi_k,torch.zeros_like(phi_k)),query_shift


def fixed_kernel_attention(q,k,v,valid,positions=None,feature_matrix=None,tau=None,
                           indices=None,grid=None,local_radius=1,return_stats=False):
    """O(B H N R D + B H N L(R+D)) finite-feature attention, L<=9.

    Numerator and denominator use the same positive approximate FAR kernel.
    Predeclared local entries replace that kernel by exp(q_i k_j/tau) exactly.
    Global application uses batched matrix products, with no N*N far tensor.
    Numerical cancellation/underflow is refused, never silently clamped.
    positions is accepted for API alignment; grid topology uses original IDs.
    """
    Q,K,V,W,temperature=_validate(q,k,v,valid,feature_matrix,tau);B,H,N,D=Q.shape;rank=W.shape[1]
    if int(local_radius)>=0:
        if indices is None or grid is None:raise ValueError('Exact local replacement needs original indices and grid')
        neighbors,local_valid=local_neighbor_indices(indices,valid,grid,local_radius)
    else:neighbors=valid.new_zeros((B,N,0),dtype=torch.long);local_valid=valid.new_zeros((B,N,0))
    local_scores=[]
    for slot in range(neighbors.shape[-1]):
        keys=_gather_tokens(K,neighbors[...,slot]);score=(Q*keys).sum(-1)/temperature[None,:,None]
        local_scores.append(score.masked_fill(~local_valid[...,slot][:,None],-torch.inf))
    scores=torch.stack(local_scores,-1)if local_scores else Q.new_empty((B,H,N,0))
    logq,logk,key_shift=_log_features(Q,K,W,temperature,valid)
    phi_q,phi_k,query_shift=_feature_values(logq,logk,key_shift,valid,scores)
    # The key-side global summaries have R*D and R elements, independent of N^2.
    summary=phi_k.transpose(-1,-2)@V;mass=phi_k.sum(-2)
    numerator=phi_q@summary;denominator=(phi_q*mass[:,:,None]).sum(-1)
    for slot in range(neighbors.shape[-1]):
        key_features=_gather_tokens(phi_k,neighbors[...,slot]);approximate=(phi_q*key_features).sum(-1)
        exact=torch.exp(scores[...,slot]-query_shift-key_shift[:,:,None])
        delta=torch.where(local_valid[...,slot][:,None],exact-approximate,torch.zeros_like(exact))
        values=_gather_tokens(V,neighbors[...,slot]);numerator=numerator+delta[...,None]*values;denominator=denominator+delta
    relevant=denominator[valid[:,None].expand(B,H,N)]
    if not bool(torch.isfinite(relevant).all())or bool((relevant<=torch.finfo(q.dtype).tiny).any()):
        raise FloatingPointError('Unsupported nonpositive/underflowed finite-kernel denominator; gate is UNKNOWN')
    safe_denominator=torch.where(valid[:,None],denominator,torch.ones_like(denominator))
    output=(numerator/safe_denominator[...,None])*valid[:,None,:,None]
    if not bool(torch.isfinite(output).all()):raise FloatingPointError('Nonfinite finite-kernel output')
    if not return_stats:return output
    pairs=int(local_valid.sum());stats={'algorithm':BACKEND,'rank':int(rank),'head_dimension':int(D),'local_radius':int(local_radius),
        'local_pairs':pairs,'feature_elements':2*B*H*N*rank,'global_summary_elements':B*H*rank*(D+1),
        'dense_reference_entries':B*H*N*N,'exact_local_kernel_entries':H*pairs,
        'far_pair_matrix_constructed':False,'adaptive_pivots':False,'basis_sha256':basis_sha256(feature_matrix),
        'feature_state':'Checkpointed frozen basis; never redrawn per forward',
        'arithmetic_scope':'Approximate dominant multiply-adds for feature projections and contractions; not a time guarantee',
        'estimated_multiply_adds':4*B*H*N*rank*D+H*pairs*(rank+2*D),
        'kernel_scope':'Finite positive exponential random-feature kernel plus exact local entries; approximate softmax',
        'normalization_scope':'Per-query common shift plus one global per-head key shift, matched exact local scale',
        'numerical_scope':'Positive FAR/local mathematical weights; floating cancellation/underflow refused',
        'uniform_error_certificate':False}
    return output,stats


def dense_fixed_kernel_reference(q,k,v,valid,positions=None,feature_matrix=None,tau=None,
                                 indices=None,grid=None,local_radius=1,return_weights=False):
    """Independent explicit finite-kernel matrix, correctness reference only.

    Dense N*N construction occurs HERE ONLY, never in the fast implementation.
    It replaces local entries directly, rather than global-sum subtraction.
    """
    Q,K,V,W,temperature=_validate(q,k,v,valid,feature_matrix,tau);B,H,N,D=Q.shape
    scores=(Q@K.transpose(-1,-2))/temperature[None,:,None,None]
    pair_valid=valid[:,None,:,None]&valid[:,None,None,:]
    if int(local_radius)>=0:
        if indices is None or grid is None:raise ValueError('Reference requires original indices/grid')
        height,width=map(int,grid);safe=indices.clamp(0,height*width-1);row,col=safe//width,safe%width
        local=(row[:,:,None]-row[:,None,:]).abs()<=int(local_radius)
        local=local&((col[:,:,None]-col[:,None,:]).abs()<=int(local_radius))&valid[:,:,None]&valid[:,None,:]
        local_scores=scores.masked_fill(~local[:,None],-torch.inf)
    else:local=valid.new_zeros((B,N,N));local_scores=None
    logq,logk,key_shift=_log_features(Q,K,W,temperature,valid)
    phi_q,phi_k,query_shift=_feature_values(logq,logk,key_shift,valid,local_scores)
    approximate=phi_q@phi_k.transpose(-1,-2)
    exact=torch.exp(scores.masked_fill(~local[:,None],-torch.inf)-query_shift[...,None]-key_shift[:,:,None,None])
    weights=torch.where(local[:,None],exact,approximate);weights=torch.where(pair_valid,weights,torch.zeros_like(weights))
    denominator=weights.sum(-1);safe_denominator=torch.where(valid[:,None],denominator,torch.ones_like(denominator))
    output=(weights@V)/safe_denominator[...,None]
    output=output*valid[:,None,:,None]
    return (output,weights)if return_weights else output


def exact_softmax_attention(q,k,v,valid,positions=None,tau=None,**unused):
    """Optimized SDPA reference for content-only ViT; not the finite kernel."""
    H,D=q.shape[1],q.shape[-1];temperature=torch.full((H,),math.sqrt(D),device=q.device,dtype=q.dtype)if tau is None else torch.as_tensor(tau,device=q.device,dtype=q.dtype)
    mask=valid[:,None,:,None];Q,K,V=[torch.where(mask,t,torch.zeros_like(t))for t in(q,k,v)]
    output=F.scaled_dot_product_attention(Q*(math.sqrt(D)/temperature)[None,:,None,None],K,V,attn_mask=valid[:,None,None,:],dropout_p=0.)
    return torch.where(mask,output,torch.zeros_like(output))


class FixedKernelMixer(nn.Module):
    """Shared ViT projection names; only content interaction application differs.

    Signed spatial bias is disabled explicitly. Nonzero incoming spatial bias
    is refused, not silently dropped. Frozen geometry scaffolding stays named
    consistently for diagnostics and old model building interfaces.
    """
    def __init__(self,reference,options=None):
        super().__init__();options=dict(options or{});allowed={'rank','feature_seed','local_radius','spatial_mode'}
        if set(options)-allowed:raise ValueError('Unknown fixed-kernel options:'+str(sorted(set(options)-allowed)))
        self.dim=int(reference.dim);self.heads=int(reference.heads);self.rank=int(options.get('rank',64));self.feature_seed=int(options.get('feature_seed',20261717));self.local_radius=int(options.get('local_radius',1))
        if self.rank not in ALLOWED_RANKS or self.local_radius not in(-1,0,1):raise ValueError('Ranks32/64/128 and declared local radius -1/0/1 only')
        if options.get('spatial_mode','disabled')!='disabled':raise ValueError('This hypothesis is content-only; spatial_mode must be disabled')
        for name,module in reference.named_children():self.add_module(name,copy.deepcopy(module))
        for name,param in reference._parameters.items():
            if param is not None:self.register_parameter(name,nn.Parameter(param.detach().clone(),requires_grad=param.requires_grad))
        for name,value in reference._buffers.items():
            if value is not None:self.register_buffer(name,value.detach().clone())
        device,dtype=self.qkv.weight.device,self.qkv.weight.dtype
        if not hasattr(self,'log_temperature_multiplier'):self.log_temperature_multiplier=nn.Parameter(torch.zeros(self.heads,device=device,dtype=dtype),requires_grad=False)
        if not hasattr(self,'spatial_coefficient_raw'):self.spatial_coefficient_raw=nn.Parameter(torch.zeros(self.heads,device=device,dtype=dtype),requires_grad=False)
        if bool((self.spatial_coefficient_raw.detach()!=0).any()):raise ValueError('Nonzero incoming spatial coefficient cannot be discarded')
        self.spatial_coefficient_raw.requires_grad_(False)
        if not hasattr(self,'log_spatial_length'):self.log_spatial_length=nn.Parameter(torch.full((self.heads,),math.log(.35),device=device,dtype=dtype),requires_grad=False)
        self.log_spatial_length.requires_grad_(False);self.spatial_coefficient_bound=getattr(reference,'spatial_coefficient_bound',2.)
        self.register_buffer('feature_matrix',make_orthogonal_features(self.heads,self.dim//self.heads,self.rank,self.feature_seed,dtype=dtype).to(device))
        self.backend=BACKEND;self.last_operator_stats={}
    def geometry_parameters(self,dtype=None):
        dtype=dtype or self.qkv.weight.dtype;multiplier=self.log_temperature_multiplier.to(dtype).clamp(-math.log(16.),math.log(16.)).exp()
        return {'temperature':math.sqrt(self.dim//self.heads)*multiplier,'temperature_multiplier':multiplier,
                'spatial_coefficient':torch.zeros_like(multiplier),'spatial_length':self.log_spatial_length.to(dtype).exp()}
    def _state(self,x,indices,valid,grid):
        if valid is None:valid=torch.ones(x.shape[:2],device=x.device,dtype=torch.bool)
        if indices is None or grid is None:raise ValueError('Original patch indices/grid are required')
        safe=torch.where(valid[...,None],x,torch.zeros_like(x));B,N,D=x.shape
        q,k,v=self.qkv(safe).reshape(B,N,3,self.heads,D//self.heads).permute(2,0,3,1,4).unbind(0)
        geo=self.geometry_parameters(q.dtype);position=torch.stack(((indices.clamp_min(0)//grid[1]+.5)/grid[0],(indices.clamp_min(0)%grid[1]+.5)/grid[1]),-1).to(x.dtype)
        return q,k,v,valid,geo,position
    def forward(self,x,indices=None,valid=None,grid=None):
        q,k,v,valid,geo,positions=self._state(x,indices,valid,grid)
        output=fixed_kernel_attention(q,k,v,valid,positions,self.feature_matrix,geo['temperature'],indices,grid,self.local_radius)
        output=output.transpose(1,2).reshape_as(x);return self.projection(output)*valid[...,None]
    def kernel_state(self,x,indices,valid,grid):
        q,k,v,valid,geo,positions=self._state(x,indices,valid,grid)
        return {'q':q,'k':k,'v':v,'query':q,'key':k,'value':v,'indices':indices,'valid':valid,'grid':grid,'positions':positions,
                'feature_matrix':self.feature_matrix,'rank':self.rank,**geo,'geometry':'Finite positive content kernel; signed spatial bias disabled',
                'diagnostic_scope':'Raw Q/K/V and frozen feature basis only; no dense diagnostic matrix in model forward'}


def replace_encoder_mixers(model,options=None):
    options=dict(options or{});seed=int(options.get('feature_seed',20261717))
    if len(model.context.blocks)!=len(model.target.blocks):raise ValueError('Context/teacher depth differs')
    for layer,(context,target)in enumerate(zip(model.context.blocks,model.target.blocks)):
        current={**options,'feature_seed':seed+layer};context.mixer=FixedKernelMixer(context.mixer,current);target.mixer=FixedKernelMixer(target.mixer,current)
        if not torch.equal(context.mixer.feature_matrix,target.mixer.feature_matrix):raise AssertionError('Teacher/context feature basis differs')
    model.target.requires_grad_(False).eval();return model